# Telecom Customer Churn Prediction - Machine Learning

**Author:** Mahesh Thakare  
**GitHub:** https://github.com/mahesh735-ai  
**LinkedIn:** https://www.linkedin.com/in/mahesh-thakare-75817b2a7

---

**Project Overview**

This notebook is Part 2 of the Telecom Customer Churn project.  
Part 1 covered Exploratory Data Analysis (EDA) on 7,043 customer records across 21 features.

In this notebook, I am building a Machine Learning model to predict which customers are likely to churn.  
The goal is not just to get a good accuracy score, but to actually identify high-risk customers  
so the business can take targeted retention actions.

**Approach:**
- Data cleaning and preprocessing
- Feature engineering to extract better signals
- Handling class imbalance using SMOTE
- Training a Logistic Regression model
- Evaluating using Precision, Recall, F1-Score and ROC-AUC
- Identifying high-risk customers using probability scores

**Tools used:** Python, Pandas, Scikit-Learn, imbalanced-learn (SMOTE), Matplotlib, Seaborn


## Step 1 - Import Libraries

Importing all required libraries upfront.  
The key addition compared to the EDA notebook is Scikit-Learn for modeling  
and imbalanced-learn for handling the class imbalance problem.


In [ ]:
# Standard data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn - preprocessing and model
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay
)

# SMOTE for handling class imbalance
from imblearn.over_sampling import SMOTE

print("All libraries imported successfully.")

## Step 2 - Load the Dataset

Loading the same dataset used in the EDA notebook.  
7,043 customer records, 21 columns including demographics, services, contract type, billing, and the target variable Churn.


In [ ]:
# Load dataset
# If running on Google Colab - upload the CSV file manually or mount Google Drive
df = pd.read_csv('Customer_Churn.csv')

print("Dataset loaded.")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
df.head()

In [ ]:
# Quick check on target variable distribution
print("Churn distribution:")
print(df['Churn'].value_counts())
print()
print(f"Churn rate: {df['Churn'].value_counts(normalize=True)['Yes']*100:.1f}%")

# This confirms the class imbalance issue we need to handle
# 73.5% No-Churn vs 26.5% Churn

## Step 3 - Data Cleaning and Preprocessing

A few issues were identified during EDA that need to be fixed before modeling:

1. **TotalCharges** is stored as object dtype due to blank spaces in some rows - needs to be converted to numeric
2. **SeniorCitizen** is stored as 0 and 1 integers - converting to No/Yes for consistency with other binary columns
3. **customerID** is just a unique identifier and has no predictive value - will be dropped

These are the same findings from Part 1 (EDA), now being applied before feeding data into the model.


In [ ]:
# Fix TotalCharges column
# Blank spaces cause it to be read as object - convert to numeric
# Rows with blanks will become NaN, then filled with median

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing_before = df['TotalCharges'].isnull().sum()
print(f"Rows with missing TotalCharges after conversion: {missing_before}")

# Fill missing values with median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

print(f"Missing values after filling: {df['TotalCharges'].isnull().sum()}")
print(f"TotalCharges dtype is now: {df['TotalCharges'].dtype}")

In [ ]:
# Convert SeniorCitizen from 0/1 to No/Yes
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

# Drop customerID - not a feature
df.drop(columns=['customerID'], inplace=True)

print(f"Dataset shape after cleaning: {df.shape}")
print()
print("Null values in dataset:")
print(df.isnull().sum().sum(), "total null values")

## Step 4 - Feature Engineering

Raw columns sometimes do not give the model the best signal.  
I am creating three new features based on domain understanding of the churn problem.

**tenure_group** - Instead of raw tenure in months, grouping customers into lifecycle stages.  
A customer in their first year behaves very differently from a 4+ year customer.

**has_streaming** - Whether a customer uses streaming TV or movies (or both).  
Customers using entertainment services may have different churn behavior.

**total_services** - Count of add-on services subscribed.  
More services means more switching cost, which typically reduces churn.


In [ ]:
# Feature 1 - Tenure Groups
# Grouping tenure (months) into customer lifecycle stages
# This was one of the key findings in EDA - Year 1 had 50% churn rate

def assign_tenure_group(tenure):
    if tenure <= 12:
        return '0_1_Year'
    elif tenure <= 24:
        return '1_2_Years'
    elif tenure <= 48:
        return '2_4_Years'
    else:
        return '4_Plus_Years'

df['tenure_group'] = df['tenure'].apply(assign_tenure_group)

print("Tenure group distribution:")
print(df['tenure_group'].value_counts())

In [ ]:
# Feature 2 - Has Streaming (TV or Movies or both)
df['has_streaming'] = (
    (df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')
).astype(int)

# Feature 3 - Total add-on services count
# Counts how many of these 6 services the customer has subscribed to
addon_services = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

df['total_services'] = df[addon_services].apply(
    lambda row: (row == 'Yes').sum(), axis=1
)

print("New features preview:")
print(df[['tenure', 'tenure_group', 'has_streaming', 'total_services']].head(10))

## Step 5 - Encoding Categorical Variables

Logistic Regression (and most ML models) require numeric inputs.  
All categorical columns need to be converted to numbers.

I am using Label Encoding here since most columns are binary (Yes/No) or have few categories.  
The target variable Churn is mapped explicitly: No = 0, Yes = 1.


In [ ]:
# Encode target variable explicitly
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

print("Target variable after encoding:")
print(df['Churn'].value_counts())
print(f"Churn rate: {df['Churn'].mean()*100:.1f}%")

In [ ]:
# Label encode all remaining object columns
le = LabelEncoder()

categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Columns to encode: {categorical_cols}")
print()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print("Encoding complete.")
print(f"Dataset shape: {df.shape}")
print()
print("All dtypes now numeric:")
print(df.dtypes.value_counts())

## Step 6 - Handling Class Imbalance with SMOTE

From EDA we already know the dataset is imbalanced - 73.5% No-Churn vs 26.5% Churn.

If we train a model on this directly, it will learn to mostly predict No-Churn  
because that gives high accuracy without actually learning anything useful.  
The model will miss the actual churners which defeats the entire purpose.

**SMOTE - Synthetic Minority Over-sampling Technique**  
Creates synthetic (not just duplicate) samples of the minority class by interpolating  
between existing minority class points. This gives the model a balanced view of both classes.

I am applying SMOTE only on training data, not test data.  
The test set must reflect real-world distribution to give honest evaluation results.


In [ ]:
# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

print("Before SMOTE:")
print(f"Class 0 (No Churn): {(y == 0).sum()}")
print(f"Class 1 (Churn)   : {(y == 1).sum()}")
print(f"Imbalance ratio   : {(y==0).sum() / (y==1).sum():.2f}:1")

In [ ]:
# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("After SMOTE:")
print(f"Class 0 (No Churn): {(y_resampled == 0).sum()}")
print(f"Class 1 (Churn)   : {(y_resampled == 1).sum()}")
print(f"Total samples     : {len(X_resampled)}")

In [ ]:
# Visualize class distribution before and after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Before SMOTE
before_counts = y.value_counts()
axes[0].bar(
    ['No Churn', 'Churn'],
    before_counts.values,
    color=['#5B9BD5', '#ED7D31'],
    edgecolor='black',
    width=0.45
)
axes[0].set_title('Before SMOTE', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(before_counts.values):
    axes[0].text(i, v + 40, f'{v} ({v/len(y)*100:.1f}%)',
                 ha='center', fontsize=10, fontweight='bold')

# After SMOTE
after_counts = pd.Series(y_resampled).value_counts().sort_index()
axes[1].bar(
    ['No Churn', 'Churn'],
    after_counts.values,
    color=['#5B9BD5', '#70AD47'],
    edgecolor='black',
    width=0.45
)
axes[1].set_title('After SMOTE (Balanced)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Number of Customers')
for i, v in enumerate(after_counts.values):
    axes[1].text(i, v + 40, f'{v} (50.0%)',
                 ha='center', fontsize=10, fontweight='bold')

fig.suptitle('Class Distribution Before and After SMOTE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 - Train-Test Split and Feature Scaling

Splitting data 80% train and 20% test with stratify to maintain class proportion in both splits.

**Why StandardScaler?**  
Logistic Regression uses gradient-based optimization internally.  
If features have very different scales (e.g. MonthlyCharges 20-100 vs tenure 1-72),  
the model will be biased towards features with larger values.  
StandardScaler brings all features to mean=0 and std=1.

Important: The scaler is fit only on training data.  
The same fitted scaler is then used to transform the test data.  
This prevents data leakage from the test set into training.


In [ ]:
# Train-Test Split - 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.2,
    random_state=42,
    stratify=y_resampled
)

print(f"Training set  : {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test set      : {X_test.shape[0]} samples")
print()
print(f"Train churn rate: {y_train.mean()*100:.1f}%")
print(f"Test churn rate : {y_test.mean()*100:.1f}%")

In [ ]:
# Feature Scaling
scaler = StandardScaler()

# Fit on training data only, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Feature scaling done.")
print(f"X_train mean (should be ~0): {X_train_scaled.mean():.4f}")
print(f"X_train std  (should be ~1): {X_train_scaled.std():.4f}")

## Step 8 - Logistic Regression Model Training

**Why Logistic Regression?**

For a classification problem like churn prediction, Logistic Regression is a strong starting point because:
- It is interpretable - we can see which features are driving the prediction
- It outputs probability scores, not just Yes/No - useful for ranking customers by risk level
- It trains fast on this dataset size
- It is commonly used and well understood in business analytics

Parameters used:
- max_iter=1000 to ensure convergence
- class_weight='balanced' as an extra safeguard against imbalance
- solver='lbfgs' which works well for binary classification


In [ ]:
# Train Logistic Regression model
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',
    solver='lbfgs'
)

lr_model.fit(X_train_scaled, y_train)

# Predictions on test set
y_pred       = lr_model.predict(X_test_scaled)
y_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

print("Model training complete.")
print()
print(f"Training Accuracy : {lr_model.score(X_train_scaled, y_train)*100:.2f}%")
print(f"Test Accuracy     : {lr_model.score(X_test_scaled, y_test)*100:.2f}%")

## Step 9 - Model Evaluation

Accuracy alone is a misleading metric for churn prediction.  
If the model just predicts No-Churn for everyone it would get 73.5% accuracy - which is useless.

The metrics that actually matter here:

**Precision** - Out of all customers the model flagged as churners, how many actually churned?  
**Recall** - Out of all actual churners, how many did the model catch?  
**F1-Score** - Harmonic mean of Precision and Recall. Good overall balance metric.  
**ROC-AUC** - How well can the model separate churners from non-churners across all thresholds.

For churn prediction, Recall is the most important.  
Missing an actual churner (False Negative) is more costly to the business  
than flagging a non-churner as a churner (False Positive).


In [ ]:
# Classification report
print("Classification Report - Logistic Regression")
print("-" * 50)
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score : {roc_auc:.4f}")

In [ ]:
# Confusion Matrix and ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix - Logistic Regression', fontsize=12, fontweight='bold')

# Label the four quadrants
labels = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        axes[0].text(j, i - 0.32, labels[i][j],
                     ha='center', fontsize=10, color='firebrick', fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='#1F4E79', lw=2,
             label=f'Logistic Regression  AUC = {roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1.2, label='Random Classifier  AUC = 0.500')
axes[1].fill_between(fpr, tpr, alpha=0.07, color='#1F4E79')
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate (Recall)', fontsize=11)
axes[1].set_title('ROC Curve - Logistic Regression', fontsize=12, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10 - Feature Importance (Which Factors Drive Churn?)

In Logistic Regression, the model coefficients tell us the influence of each feature.

A positive coefficient means that feature increases the probability of churn.  
A negative coefficient means that feature reduces the probability of churn.

This is the business insight layer of the project.  
It answers the question: where exactly should the company focus its retention efforts?


In [ ]:
# Extract feature coefficients
feature_names = X.columns.tolist()
coefficients  = lr_model.coef_[0]

feat_df = pd.DataFrame({
    'Feature'    : feature_names,
    'Coefficient': coefficients,
    'Abs_Value'  : np.abs(coefficients)
}).sort_values('Abs_Value', ascending=False).reset_index(drop=True)

print("Top 15 most influential features:")
print(feat_df.head(15).to_string(index=False))

In [ ]:
# Plot feature importance
top15 = feat_df.head(15).sort_values('Coefficient')

bar_colors = ['#C00000' if c > 0 else '#1F4E79' for c in top15['Coefficient']]

plt.figure(figsize=(10, 6))
plt.barh(top15['Feature'], top15['Coefficient'],
         color=bar_colors, edgecolor='black', height=0.6)
plt.axvline(x=0, color='black', linewidth=1.0, linestyle='--')
plt.xlabel('Logistic Regression Coefficient', fontsize=11)
plt.title(
    'Feature Importance - Logistic Regression\n'
    'Red = Increases Churn Risk    |    Blue = Reduces Churn Risk',
    fontsize=12, fontweight='bold'
)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("Top churn risk drivers (positive coefficients):")
print(feat_df[feat_df['Coefficient'] > 0][['Feature', 'Coefficient']].head(5).to_string(index=False))
print()
print("Top retention factors (negative coefficients):")
print(feat_df[feat_df['Coefficient'] < 0][['Feature', 'Coefficient']].head(5).to_string(index=False))

## Step 11 - Identifying High Risk Customers

The real business output of this model is not the accuracy score.  
It is a ranked list of customers by churn probability.

The retention team can use this list to prioritize who to call, what offer to make,  
and where to spend the retention budget most efficiently.

I am applying the trained model on the original pre-SMOTE data  
to score all 7,043 real customers with their individual churn probability.

Risk segments:
- 0 to 30% probability - Low Risk
- 30 to 50% probability - Medium Risk
- 50 to 70% probability - High Risk
- 70% and above - Critical Risk (immediate action needed)


In [ ]:
# Score all original customers using the trained model
X_original        = df.drop('Churn', axis=1)
X_original_scaled = scaler.transform(X_original)

churn_probability = lr_model.predict_proba(X_original_scaled)[:, 1]

# Build customer risk report
risk_df = df.copy()
risk_df['Churn_Probability_Pct'] = (churn_probability * 100).round(2)
risk_df['Risk_Segment'] = pd.cut(
    churn_probability,
    bins=[0, 0.30, 0.50, 0.70, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk']
)

print("Customer Risk Segmentation Summary:")
print(risk_df['Risk_Segment'].value_counts())
print()
critical_count = (risk_df['Risk_Segment'] == 'Critical Risk').sum()
high_count     = (risk_df['Risk_Segment'] == 'High Risk').sum()
print(f"Customers needing immediate retention action (Critical): {critical_count}")
print(f"Customers needing follow-up (High Risk)               : {high_count}")

In [ ]:
# Show top 10 highest risk customers
print("Top 10 highest churn probability customers:")
risk_df.sort_values('Churn_Probability_Pct', ascending=False)[
    ['tenure', 'Contract', 'MonthlyCharges', 'Churn_Probability_Pct', 'Risk_Segment']
].head(10)

In [ ]:
# Risk segment distribution - bar chart
risk_counts = risk_df['Risk_Segment'].value_counts().reindex(
    ['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk']
)

bar_colors = ['#375623', '#7F6000', '#C55A11', '#C00000']

plt.figure(figsize=(9, 5))
bars = plt.bar(risk_counts.index, risk_counts.values,
               color=bar_colors, edgecolor='black', width=0.5)
plt.title('Customer Churn Risk Segmentation\nBased on Logistic Regression Probability Scores',
          fontsize=12, fontweight='bold')
plt.xlabel('Risk Segment', fontsize=11)
plt.ylabel('Number of Customers', fontsize=11)
plt.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, risk_counts.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{val}
({val/len(risk_df)*100:.1f}%)',
             ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('risk_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 12 - Project Summary and Business Recommendations

### Model Performance

The Logistic Regression model was trained on SMOTE-balanced data and evaluated on the original test distribution.  
Classification report and ROC-AUC scores are printed in Step 9 above.

Key observation: A high Recall score on the Churn class means the model is successfully  
catching most actual churners - which is the primary objective of this project.

---

### Business Recommendations Based on Model Findings

**1. Month-to-month contract customers are the highest risk group.**  
The model confirms this is the strongest churn predictor.  
The company should offer incentives to convert these customers to annual or two-year plans.

**2. Fiber Optic internet users churn despite paying premium prices.**  
This suggests a service quality or value-for-money issue.  
Targeted service improvement or pricing adjustments are needed for this segment.

**3. The first 12 months are the most critical retention window.**  
tenure_group is one of the top features. 50% of churned customers left in Year 1.  
A structured onboarding program and proactive support in the first year could significantly reduce this.

**4. Electronic check users have 45% churn vs 15% for auto-pay users.**  
Migrating customers to auto-pay (bank transfer or credit card) reduces churn.  
A small bill credit as incentive to switch payment method could have high ROI.

**5. Use the probability scores from Step 11 for targeted campaigns.**  
Rather than treating all customers the same, the retention team should focus  
effort and budget on Critical Risk and High Risk segments first.

---

### Files in this Repository

- Customer_Churn.csv - Raw dataset
- Churn_analysis_EDA_P2.ipynb - Part 1: Exploratory Data Analysis
- ML_churn_analysis.ipynb - Part 2: Machine Learning (this notebook)

---

Mahesh Thakare  
GitHub: https://github.com/mahesh735-ai  
LinkedIn: https://www.linkedin.com/in/mahesh-thakare-75817b2a7
